# Philosophy Quote Generator

#### Base:
https://dzone.com/articles/build-a-philosophy-quote-generator-with-vector-sea

https://dzone.com/articles/infinite-wisdom-series-build-a-philosophy-quote-ge

https://dzone.com/articles/build-a-philosophy-quote-generator-with-vector-sea-1

## Initialization

In [1]:
import chromadb
import json
import openai

from openai import OpenAI

from chromadb.utils import embedding_functions
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()


True

### Variables

In [1]:
embedding_model_name = "all-mpnet-base-v2" # text-embedding-ada-002
openAI_embedding_model_name = "text-embedding-ada-002"

chroma_user = os.getenv("CHROMA_USER")
chroma_pass = os.getenv("CHROMA_PASS")
chroma_host = os.getenv("CHROMA_HOST")
chroma_port = os.getenv("CHROMA_PORT")

n_results = 5

user_query = "Solo sé que no sé nada"
query = [user_query]

NameError: name 'os' is not defined

In [4]:
print(chroma_user)
print(chroma_pass)
print(chroma_host)
print(chroma_port)

chroma
lueBAQBbCZZOiHLj
chromadb.reto-ucu.net
50004


In [5]:
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=embedding_model_name)
# https://sbert.net/docs/sentence_transformer/pretrained_models.html

C:\Users\Admin\anaconda3\envs\rag-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
openai_ef = OpenAIEmbeddingFunction(
    model_name = openAI_embedding_model_name
)

In [29]:
from chromadb import Client
from chromadb.config import Settings

import base64

auth_string = f"{chroma_user}:{chroma_pass}"
auth_header = {
    "Authorization": "Basic " + base64.b64encode(auth_string.encode()).decode()
}

settings = Settings(
    chroma_api_impl="rest",
    chroma_server_host=chroma_host,
    chroma_server_http_port=chroma_port,
    chroma_server_headers=auth_header
)

chroma_client = Client(settings)

ValueError: Unsupported Chroma API implementation rest

In [8]:
collection = chroma_client.get_or_create_collection(
    name = "philosphy_quotes23",
    embedding_function = sentence_transformer_ef
)

In [9]:
collectionOpenAI = chroma_client.get_or_create_collection(
    name = "philosophy_quotes_ada",
    embedding_function = openai_ef
)

## Read file

In [10]:
with open('files/quotes.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

    quotes = []
    metadatas = []
    ids = []
    id = 1

    for quote in data["quotes"]:
        quotes.append(quote["quote"])
        metadatas.append({"author": quote["author"]})
        ids.append(str(id))

        id+=1

In [11]:
quotes

['La pluma es la lengua del alma.',
 'El que quiere interesar a los demás tiene que provocarlos.',
 'Todos los niños nacen artistas. El problema es cómo seguir siendo artistas al crecer.',
 'Una es más auténtica, mientras más se parece a lo que soñó de sí misma.',
 'Soy el desesperado, la palabra sin ecos, el que lo perdió todo, y el que todo lo tuvo.',
 'Aprender a sonreír es aprender a ser libres.',
 'Memoria selectiva para recordar lo bueno, prudencia lógica para no arruinar el presente, y optimismo desafiante para encarar el futuro.',
 '¿Se pueden inventar verbos? quiero decirte uno: Yo te cielo, así mis alas se extienden enormes para amarte sin medida.',
 'Se necesitan dos años para aprender a hablar y sesenta para aprender a callar.',
 'No hay que ir para atrás ni para darse impulso.',
 'No hay caminos para la paz; la paz es el camino.',
 'Haz el amor y no la guerra.',
 'Para trabajar basta estar convencido de una cosa: que trabajar es menos aburrido que divertirse.',
 'Lo peor q

In [12]:
collection.add(
    ids = ids,
    metadatas = metadatas,
    documents = quotes
)

In [13]:
collectionOpenAI.add(
    ids = ids,
    metadatas = metadatas,
    documents = quotes
)

## Some trials

In [14]:
quote_results = collection.query(
    query_texts = query,
    n_results = n_results,
    include = ["distances", "metadatas", "documents"]
)

# quote_results

In [15]:
quote_resultsOpenAI = collectionOpenAI.query(
    query_texts = query,
    n_results = n_results,
    include = ["distances", "metadatas", "documents"]
)

quote_resultsOpenAI

{'ids': [['69', '17', '35', '5', '63']],
 'embeddings': None,
 'documents': [['Solo sé que no sé nada.',
   'Cada día sabemos más y entendemos menos.',
   'Hay dos cosas que son infinitas: el universo y la estupidez humana; de la primera no estoy muy seguro.',
   'Soy el desesperado, la palabra sin ecos, el que lo perdió todo, y el que todo lo tuvo.',
   'Es mejor permanecer callado y parecer tonto que hablar y despejar las dudas definitivamente.']],
 'uris': None,
 'included': ['distances', 'metadatas', 'documents'],
 'data': None,
 'metadatas': [[{'author': 'Sócrates'},
   {'author': 'Albert Einstein'},
   {'author': 'Albert Einstein'},
   {'author': 'Pablo Neruda'},
   {'author': 'Groucho Marx'}]],
 'distances': [[0.04672810062766075,
   0.3631477355957031,
   0.3946390151977539,
   0.3996537923812866,
   0.4025905430316925]]}

In [16]:
client = chromadb.PersistentClient(path="vectordb")

## Quote Generator

In [17]:
prompt = ""
completion_model_name = "gpt-3.5-turbo"
client = OpenAI()

In [18]:
generation_prompt_template = """"Genera una sencilla y corta frase filosófica, dada la siguiente frase de referencia, similar en espíritu, y usando como base los ejemplos
No te excedas de 20 o 30 palabras

FRASE DE REFERENCIA: "{topic}"


EJEMPLOS: "{examples}"

"""


In [19]:
def find_quote_and_author_p(collection, query, n_results, author=None, tags=None):
    quote_results = collection.query(
        query_texts = query,
        n_results = n_results,
        include = ["distances", "metadatas", "documents"]
    )

    return quote_results

In [20]:
def generate_quote(collection, query, n_results=2, author=None, tags=None):
    quotes = find_quote_and_author_p(
        collection = collection,
        query = query,
        n_results = n_results,
        author = author,
        tags = tags
    )

    if quotes:
        prompt = generation_prompt_template.format(
            topic = query[0],
            examples = "\n".join(f"  - {quote[0]}" for quote in quotes),
        )

        response = client.chat.completions.create(
            model=completion_model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
            max_tokens=320
        )

        return response.choices[0].message.content.replace('"', '').strip()

    else:
        print("** no quotes found.")
        return None

In [21]:
generate_quote(collection, query, n_results)

'En la búsqueda de conocimiento, descubrimos nuestra ignorancia'

In [22]:
generate_quote(collectionOpenAI, query, n_results)

'En la incertidumbre reside la sabiduría verdadera'